# Customer Segmentation and Explainable Churn-Risk Model
### Task 3: Behavioral Feature Engineering, Rule-Based Scoring Engine, Edge-Case Auditing, and Retention Playbook

**Track:** Data Analytics | **Level:** Beginner-Friendly  
**Objective:** Move from descriptive analytics to an explainable prioritization model by engineering behavioral and account features, defining customer cohorts, designing a transparent rule-based score (0-100), validating edge cases, and prescribing retention actions for high-value segments.

## 1. Data Ingestion & Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ingest clean verified dataset
df = pd.read_csv('customer_churn_cleaned.csv')
print(f"Loaded {len(df)} records.")

# Engineer derived behavioral and account features
df['ticket_velocity'] = (df['support_tickets'] / np.maximum(df['tenure_months'], 1)).round(3)
df['annual_value'] = (df['monthly_charge'] * 12).round(2)
df[['customer_id', 'contract_type', 'tenure_months', 'support_tickets', 'ticket_velocity', 'monthly_charge', 'annual_value', 'churned']].head()

## 2. Transparent Rule-Based Churn Risk Scoring Engine

The model computes an explainable **Churn-Risk Score (0 to 100 points)**:
- **Contract Score (Max 40 pts):** Month-to-Month = 40, One Year = 10, Two Year = 0
- **Support Ticket Score (Max 35 pts):** >=5 = 35, 3-4 = 25, 2 = 10, 0-1 = 0
- **Tenure Score (Max 15 pts):** 0-6m = 15, 7-12m = 10, 13-24m = 5, 25+m = 0
- **Payment Score (Max 10 pts):** UPI/Debit Card = 10, Credit Card = 5, Bank Transfer = 0

In [2]:
def score_contract(c):
    return 40 if c == 'Month-to-Month' else (10 if c == 'One Year' else 0)

def score_tickets(t):
    return 35 if t >= 5 else (25 if t >= 3 else (10 if t == 2 else 0))

def score_tenure(m):
    return 15 if m <= 6 else (10 if m <= 12 else (5 if m <= 24 else 0))

def score_payment(p):
    return 10 if p in ['UPI', 'Debit Card'] else (5 if p == 'Credit Card' else 0)

def calculate_risk(row):
    c_pts = score_contract(row['contract_type'])
    t_pts = score_tickets(row['support_tickets'])
    m_pts = score_tenure(row['tenure_months'])
    p_pts = score_payment(row['payment_method'])
    total = c_pts + t_pts + m_pts + p_pts
    tier = 'Critical Risk' if total >= 75 else ('Elevated Risk' if total >= 50 else ('Moderate Risk' if total >= 25 else 'Low Risk'))
    return pd.Series([c_pts, t_pts, m_pts, p_pts, total, tier], 
                     index=['contract_score', 'ticket_score', 'tenure_score', 'payment_score', 'churn_risk_score', 'risk_tier'])

scoring_results = df.apply(calculate_risk, axis=1)
df_scored = pd.concat([df, scoring_results], axis=1)
df_scored[['customer_id', 'contract_type', 'support_tickets', 'tenure_months', 'churn_risk_score', 'risk_tier', 'churned']]

## 3. Customer Segmentation & Cohort Definition

In [3]:
def assign_segment(row):
    if row['contract_type'] == 'Month-to-Month' and row['support_tickets'] >= 3:
        return 'Segment 1: High-Friction Flight Risks'
    elif row['contract_type'] in ['One Year', 'Two Year'] and row['support_tickets'] >= 2:
        return 'Segment 2: Contracted Friction Watchlist'
    elif row['subscription_type'] == 'Enterprise':
        return 'Segment 3: Enterprise Strategic Anchors'
    else:
        return 'Segment 4: Stable Core Subscribers'

df_scored['segment_name'] = df_scored.apply(assign_segment, axis=1)

segment_summary = df_scored.groupby('segment_name').agg(
    customer_count=('customer_id', 'count'),
    avg_risk_score=('churn_risk_score', 'mean'),
    historical_churn_rate=('churn_binary', 'mean'),
    monthly_revenue=('monthly_charge', 'sum')
).reset_index()

segment_summary['historical_churn_rate'] = (segment_summary['historical_churn_rate'] * 100).round(1).astype(str) + '%'
segment_summary['monthly_revenue'] = '₹' + segment_summary['monthly_revenue'].round(2).astype(str)
segment_summary

## 4. Edge-Case Validation & Robustness Auditing
Validating extreme, boundary, and unexpected profile scenarios to verify model behavior.

In [4]:
edge_cases = pd.DataFrame([
    {
        'test_id': 'EDGE-01',
        'desc': 'High Tickets on 2-Year Enterprise Contract',
        'contract_type': 'Two Year', 'support_tickets': 5, 'tenure_months': 36, 'payment_method': 'Bank Transfer'
    },
    {
        'test_id': 'EDGE-02',
        'desc': 'New Month-to-Month with 0 Tickets (Silent Churner)',
        'contract_type': 'Month-to-Month', 'support_tickets': 0, 'tenure_months': 1, 'payment_method': 'UPI'
    },
    {
        'test_id': 'EDGE-03',
        'desc': 'Boundary Tenure Shift (6m vs 7m)',
        'contract_type': 'Month-to-Month', 'support_tickets': 3, 'tenure_months': 6, 'payment_method': 'Credit Card'
    },
    {
        'test_id': 'EDGE-04',
        'desc': 'Veteran Enterprise Anchor (Perfect Profile)',
        'contract_type': 'Two Year', 'support_tickets': 0, 'tenure_months': 48, 'payment_method': 'Bank Transfer'
    }
])

edge_results = edge_cases.apply(calculate_risk, axis=1)
pd.concat([edge_cases[['test_id', 'desc', 'contract_type', 'support_tickets', 'tenure_months', 'payment_method']], edge_results[['churn_risk_score', 'risk_tier']]], axis=1)

## 5. Strategic Recommendations by Segment
*(Satisfies Optional Portfolio Upgrade: Recommend one retention action for each high-value segment)*

1. **Enterprise Strategic Anchors (High-Value: ₹149.99/mo):** Dedicated Technical Account Manager (TAM) and Quarterly Business Reviews (QBR).
2. **Contracted Friction Watchlist (Mid-Value Pro/Basic):** Proactive ticket resolution audit + 15% Annual Renewal rate-lock discount.
3. **High-Friction Flight Risks (Critical Hazard):** VIP concierge telephone outreach within 24h + 1-month billing credit.